In [ ]:
### Improvement over LazyVisualizeDataLazy.ipynb (v3) as we now outsource the 2D slice extraction to  
### adrienExportSlicesNetCDF.ipynb or equivalently adrienAutoExportSlicesNetCDF.py (which can be run as a batch script)
### These export the slices as NetCDF files and then the current notebook load sthem. However even this slicing in python
### (done as I do it) is pretty slow and struggles for the largest cases R4P50, R8P7, R10P7.
### ALL ROUTINES ARE IN adrienUtils.py and the case parameters and file paths are in adrienParamClassSheared.py
### Adrien Lefauve, Feburary 2026

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import importlib
from pathlib import Path

import adrienParamClassSheared as params
import adrienUtils as utils
importlib.reload(params);
importlib.reload(utils);

In [2]:
# pick a case
cases = params.generate()
p = cases["R6P7"]

# where you saved slices
slice_dir = Path(p.dirPath) / "2D_slices"
print("slice_dir:", slice_dir)

# the three files you created
paths = {
    "xy": slice_dir / "xy.nc",
    "xz": slice_dir / "xz.nc",
    "yz": slice_dir / "yz.nc",
}

planes_ok = [
    pl for pl, path in paths.items()
    if path.exists() and path.stat().st_size > 0
]

print("planes_ok:", planes_ok)

# buoyancy frequency for this run (needed for Ep and bN)
p.N2 = float(-p.dGrad * p.zAccel)
p.N  = float(np.sqrt(p.N2))

print("N2 =", p.N2)
print("N  =", p.N)

slice_dir: /lustre/orion/cfd135/proj-shared/Hsst/R6P7/001_Final/2D_slices
planes_ok: ['xy', 'xz']
N2 = 0.152492
N  = 0.3905022407106008


In [3]:
stats_by_plane = {}

for pl in planes_ok:
    path = paths[pl]
    raw, attrs = utils.read_raw_plane_netcdf(path, copy=True)
    st = utils.plane_stats(raw, p)
    stats_by_plane[pl] = st

    fixed = {"xy":"iz", "xz":"iy", "yz":"ix"}[pl]
    print(f"[{pl}] {fixed}={attrs.get(fixed, '??')}  "
          f"Ek={st['Ek']:.6e}  Ep={st['Ep']:.6e}  "
          f"eps={st['eps_avg']:.6e}  chi={st['chi_avg']:.6e}", flush=True)

# combine (simple mean across the three planes)
stats = {k: float(np.mean([stats_by_plane[pl][k] for pl in stats_by_plane]))
         for k in ["Ek","Ep","eps_avg","chi_avg"]}

print("\n[combined stats from 2D slices]")
for k, v in stats.items():
    print(f"  {k}: {v:.6e}")

Ek_slice = stats["Ek"]
Ek_pct = Ek_slice / p.targKE * 100.0

print(f"2D averaged Ek  is {Ek_pct:.2f}% of 3D average (target)")

[xy] iz=1536  Ek=2.053790e-02  Ep=2.969614e-03  eps=3.668888e-03  chi=5.948485e-04
[xz] iy=3072  Ek=1.715167e-02  Ep=2.492125e-03  eps=3.343325e-03  chi=4.899768e-04

[combined stats from 2D slices]
  Ek: 1.884478e-02
  Ep: 2.730870e-03
  eps_avg: 3.506106e-03
  chi_avg: 5.424127e-04
2D averaged Ek  is 119.34% of 3D average (target)


In [ ]:
# --- read all planes ---
raw2d = {}
meta2d = {}

def is_valid_netcdf3(path, min_bytes=256):
    path = Path(path)
    if (not path.exists()) or path.stat().st_size < min_bytes:
        return False
    with open(path, "rb") as f:
        return f.read(3) == b"CDF"   # NetCDF3 signature

# IMPORTANT: reuse the same filtering logic as earlier, but stronger than st_size>0
planes_ok = [pl for pl in ["xy", "xz", "yz"] if is_valid_netcdf3(paths[pl])]
print("planes_ok:", planes_ok)

for pl in planes_ok:
    raw2d[pl], meta2d[pl] = utils._read_plane_nc(paths[pl])
    print(pl, "loaded:", {k: raw2d[pl][k].shape for k in raw2d[pl].keys()})
    
# --- build a SliceBundle 's' containing ONLY derived fields (like get_derived_slices_multi) ---
# We reuse your SliceBundle / VarSlices classes already in adrienUtils.
out_vars = {name: utils.VarSlices(name) for name in ["uN", "vN", "wN", "bN", "epslog", "chilog"]}

Ek = stats["Ek"]
Ep = stats["Ep"]
eps_avg = stats["eps_avg"]
chi_avg = stats["chi_avg"]
N2 = float(p.N2)  # you set this earlier in Cell 5

tiny = 1e-30

for pl in raw2d.keys():
    u2 = raw2d[pl]["u"]
    v2 = raw2d[pl]["v"]
    w2 = raw2d[pl]["w"]
    r2 = raw2d[pl]["r"]
    ee2 = raw2d[pl]["ee"]
    chi2 = raw2d[pl]["chi"]

    out_vars["uN"].set(pl, u2 / np.sqrt(Ek))
    out_vars["vN"].set(pl, v2 / np.sqrt(Ek))
    out_vars["wN"].set(pl, w2 / np.sqrt(Ek))

    out_vars["bN"].set(pl, r2 * (-p.zAccel) / np.sqrt(N2 * Ep))

    with np.errstate(divide="ignore", invalid="ignore"):
        out_vars["epslog"].set(pl, np.log10(np.maximum(ee2, tiny) / eps_avg))
        out_vars["chilog"].set(pl, np.log10(np.maximum(chi2, tiny) / chi_avg))

# idx is not super meaningful here, but keep it for compatibility with plotting funcs
idx_map = {
    "xy": int(meta2d["xy"]["attrs"].get("iz", -1)) if "xy" in meta2d else -1,
    "xz": int(meta2d["xz"]["attrs"].get("iy", -1)) if "xz" in meta2d else -1,
    "yz": int(meta2d["yz"]["attrs"].get("ix", -1)) if "yz" in meta2d else -1,
}

# use any available plane to read stride (they are identical)
first_pl = next(iter(meta2d.keys()))

stride_tuple = (
    int(meta2d[first_pl]["attrs"].get("stride_x", 1)),
    int(meta2d[first_pl]["attrs"].get("stride_y", 1)),
    int(meta2d[first_pl]["attrs"].get("stride_z", 1)),
)

s = utils.SliceBundle(vars=out_vars, idx=idx_map, stride=stride_tuple)

print("planes:", s.available_planes())
print("vars:", s.available_vars())
# Check memory used by notebook
utils.memory_report(globals(), min_gb=0.01)

planes_ok: ['xy', 'xz']
xy loaded: {'u': (12288, 6144), 'v': (12288, 6144), 'w': (12288, 6144), 'r': (12288, 6144), 'ee': (12288, 6144), 'chi': (12288, 6144)}
xz loaded: {'u': (12288, 3072), 'v': (12288, 3072), 'w': (12288, 3072), 'r': (12288, 3072), 'ee': (12288, 3072), 'chi': (12288, 3072)}


In [8]:
# PLOT A SUMMARY FIGURE FOR EACH SLICE, ALL 6 VARIABLES, AT LOW RESOLUTION 
# use only planes that exist in the bundle (safe even if only one plane)
planes_avail = tuple(pl for pl in ("xy" "xz", "yz") if pl in s.available_planes())

if not planes_avail:
    raise RuntimeError("No planes available in SliceBundle 's' (nothing to plot).")

figs = utils.plot_slices_derived_bundle_multi(
    p, s,
    planes=planes_avail,
    save=True, outdir="figures", fmt="png", dpi=300
)

RuntimeError: No planes available in SliceBundle 's' (nothing to plot).

In [45]:
# PLOT FULL RESOLUTION FOR EACH VARIABLE

# stride controls resolution of output (1,1,1 = full res)
stride = (1, 1, 1)

planes_avail = tuple(pl for pl in ("xy", "xz", "yz") if pl in s.available_planes())

if not planes_avail:
    raise RuntimeError("No planes available in SliceBundle 's' (nothing to export).")

paths = utils.export_native_resolution_from_bundle_multi(
    p, s,
    planes=planes_avail,
    outdir="figures"
)

paths

{'xz': {'u': 'figures/R6P7_u_native_res_y3072_20260203_053624.png',
  'v': 'figures/R6P7_v_native_res_y3072_20260203_053624.png',
  'w': 'figures/R6P7_w_native_res_y3072_20260203_053624.png',
  'b': 'figures/R6P7_b_native_res_y3072_20260203_053624.png',
  'e': 'figures/R6P7_e_native_res_y3072_20260203_053624.png',
  'c': 'figures/R6P7_c_native_res_y3072_20260203_053624.png'}}